# Final challenge!



# Breaking the Vigenere cipher

## Kasiski examination
Despite being previously known to be unbreakable, de Vigenere cipher is suceptible to certain type of attacks. For instance, the fact that the message is encrypted with a key, makes it so that trying to use frequency analysis is not straight forward.

However, since the key repeats itself in the keystream, it's possible to find repeating patterns across a long text. For example, in the previous challenge the word 'GLZ', representing 'the' appears multiple times. In other instances, 'the' is encoded as 'XNI', which is also a 3-letter-word that appears multiple times in the code.

Looking at the distance between those repeated occurrences can give us information about the lenght of the key used. For instance, GLZ happens twice between 837 characteres, while 'XNI' happens 6 times, at intervals of: 279, 171, 144, 216, 234.

Write a function find_substring_intervals() that receives two parameters: a str text, and a str substring. Your function should return a list with the intervals in which the substring occurs in the variable text, in case it occurs more than twice.


In [1]:
def find_substring_intervals(cipher_text, sub):
  previous=cipher_text.find(sub, index)
  index = 0
  count = []
  while index < len(cipher_text):
    index = cipher_text.find(sub, index)
    if index == -1:
        break
    count.append(previous-index)
    previous=index
    index += len(sub)
  return count

Given a certain interval N in which a substring (assumed to be the same word) repeats, the length of the key L needs to be a divisor of that interval. This means that the factors of L, also need to be factors of N.

Knowing that, if we compute the factors of 279 in the example above we have the following:

[3, 3, 31]

This means that the length of the key has either the factors:
[31], [3], [3, 31], [3, 3] or [3, 3, 31].

However, we can also look at the factors of the other intervals: 171, 144, 216 and 234.

If we do that and compute the intersection of their factors, we know for sure that the factors of our key is either [3] or [3, 3].

Write a function decompose_factors() that receive an integer N and returns a list with all the factors of N. You can also create the helper function find_common_factors(), that receive a list of integers, and return the intersection of the factors of all the numbers. (Be aware that this intersection is not a set intersection, since this would remove repeating factors).




In [2]:
def decompose_factors(n):
  i = 2
  f = []
  while n!=1:
    if n%i == 0:
      f.append(i)
      n = n/i
    else:
      i+=1
  return f

def find_common_factors(intervals):
  intersect = decompose_factors(intervals[0])
  for n in intervals:
    f_n = decompose_factors(n)
    temp = intersect.copy()
    print(temp)
    for f in f_n:
      if f in temp:
        temp.remove(f)
    for remaining in temp:
      intersect.remove(remaining)

  return intersect

In [3]:

print(decompose_factors(279))
print(decompose_factors(171))
print(decompose_factors(144))
print(decompose_factors(216))
print(decompose_factors(234))
print(find_common_factors([279,171,144,216,234]))


[3, 3, 31]
[3, 3, 19]
[2, 2, 2, 2, 3, 3]
[2, 2, 2, 3, 3, 3]
[2, 3, 3, 13]
[3, 3, 31]
[3, 3, 31]
[3, 3]
[3, 3]
[3, 3]
[3, 3]


Now that we have tools to estimate the length of the cipher, we can try to break it using different strategies. The first one, is to perform frequency analysis considering only the text that was encoded using the same letter of the key.

For example, if we separate the first two sentences highlighting the letters encoded with the first letter of the key, we would have the follwoing:

>**C**SZII GMC**V**XBPJOMNX**Y** BJ BLZ **Y**IPSIL RS**X**PQ RIV. **K**RRZDMZZK TVJBNVR **A**NW JSMR **M**A 1913.

We can then assume that the subtext containing the highlighted letters were all encoded with a Caesar's cipher, and thus frequency analysis is possible.

We can then reapeat the process with the other 8 key positions, doing frequency analysis on the letters encoded with each letter of the key.


## Dictionary attack
In addition to that, if the key used is an existing word (in a known language), we can try to guess by doing a dictionary attack. This works by trying to decode with all words in the dictionary that have a similar length. In our case (assuming the key is in English), we could try to decode the text using all the 9-letter-words present in the English dictionary.

Of course this would result in many different decryption attempts. For every one of them, we could use frequency analysis to see how similar the decoded attempt is to an English text. This can be done using a frequency match score as explained below:


## Frequency match score:

To compute this frequency score, one can look for the six most common letters and the six least common letters in a cipher text, and count how many letters in those two sets intersect with the six most common and the six least common letters in English (or another languague).

For example, if a cipher text has [A, S, R, X, J, I] as the most common letters, and [F, Z, G, K, V, D] as the least common letters, the frequency match score with English would be 5:
- the intersection of [A, S, R, X, J, I] and [E, T, A, O, I, N] is [A, I]: 2
- the intersection of [F, Z, G, K, V, D] and [V, K, J, X, Q, or Z] is [Z, K, V]: 3

Implement a function to compute the frequency match score with English, given the frequency as computed in the first challenge.

In [4]:

text = "Cszii gmcVxbpjomnxy bj blz yipsil rsxpq riv. krrzdmzzk tvjbnvr anw jsmr ma 1913."
print(text.upper())
text2=[]
for i in range(0,len(text)):
  if i%9==0:
    text2.append('**')
    text2.append(text[i].upper())

    text2.append('**')
  else:
    text2.append(text[i])
print(''.join(text2).upper())

CSZII GMCVXBPJOMNXY BJ BLZ YIPSIL RSXPQ RIV. KRRZDMZZK TVJBNVR ANW JSMR MA 1913.
**C**SZII GMC**V**XBPJOMNX**Y** BJ BLZ **Y**IPSIL RS**X**PQ RIV. **K**RRZDMZZK** **TVJBNVR **A**NW JSMR **M**A 1913.


# Challenge 3 - Final challenge

Now you know of tools that can be used to break the Vigenere cipher. Try to break the encryption of the text in the next section using either frequency analysis or a dictionary attack.

(You can use the words from this [dictionary file](https://drive.google.com/file/d/1tLKTQZ2BOvC4qNLXLn42s-K8fiMAfLrh/view?usp=sharing))


In [5]:
def vigenere_cipher(plain_text, key):
  alphabet = "abcdefghijklmnopqrstuvwxyz"
  coded = ''
  plain_text=plain_text.lower()
  keystream = key*(len(plain_text)//len(key)+1)
  for i in range(len(plain_text)):
    l = plain_text[i]
    if l in alphabet:
      coded+=alphabet[(alphabet.index(l)+alphabet.index(keystream[i])) % len(alphabet)]
    else:
      coded+=l
  return coded
def decode_vigenere(text, key):
  alphabet = "abcdefghijklmnopqrstuvwxyz"
  decoded = ''
  keystream = key*(len(text)//len(key)+1)
  for i in range(len(text)):
    l = text[i]
    if l in alphabet:
      decoded+=alphabet[(alphabet.index(l)-alphabet.index(keystream[i])) % len(alphabet)]
    else:
      decoded+=l
  return decoded






TDYKGUKJWEICFCD XH RZQEFVITRV KWP UCEPW RBRAWICAV ZJ NYXD ELFVCEBGZCR FOVHE!
QV SSEY NZY BRS JJH QCIPEZCR IBV NSSYJ DS ZRG!

SFJ GPH TYXTL ISI FVPOIG SDLVS FU XWY UTRPF RSEAFVCRI VP LGRYJHTRV KWP UICAZAXHX WMCE:

XWY QZEGX LTPA ST ESPVGEMOYU LX NYT ICX DQ IBV HITE, LRS UTAICXZCR DH NZYG GDDMICFC CDO BTKWN GPGTCMT E JIXKI!

ISMH TWLPAYEVP LUJ MEHYU ZR NYT JDFCDHMCA GPWDOIRPW NYPE NIL NEC LHP II ZYSL DDCI USDFX WINAXDAIPALN:

SXIJJ://RLPEES.CEIPVPWKXGI-GRISW.WFB/

BKIAW://XYZTHKLTXWJPISSC.TDX/RLRRVMCA/


In [6]:
section = """Ada Augusta, countess of Lovelace, was the first person to ever write a program. Daughter of the skilled mathematician Annabella Milbanke with the poet Lord Byron, Ada showed enthusiasm for mathematics and science, but also arts and music. Stimulated by her mother, who sheltered her from the excesses and immoral behavior of her father, Ada grew up with a strong analytical mind. When she was 12 years old, she was inspired by the idea of flying, and extensively studied birds and aerodynamics. At 17, her potential as a mathematician started to be recognized.

In her life, she had contact with and met many important scientific figures. Among them, was Charles Babbage, whose inventions were of great interest to Ada. Ada understood the mechanisms behind Babbage’s computing engines and was fascinated by them. Eventually, Ada contributed to the translation of a text about Babage’s analytical engine and could write her own personal notes about the matter. Her notes ended up being more extensive than the original text, revolutionizing the understanding of the invention.

There she described the true potential of the analytical engine, uncovering its unforeseen use. She proposed a mechanism in which the engine could be programmed to compute different things. An example algorithm was then described in her famous note G, becoming the first program ever to be written.

Ada's legacy is even more relevant when we consider that during her lifetime, science was only accessible to the wealthiest, and mostly to men. Moreover, important women like Ada often had their work erased from history.

Nowadays, Ada inspires different initiatives fostering diversity and inclusion, such as the Ada Lovelace Day, a celebration that happens on the second Tuesday of October since 2009. Her figure is used as an inspiration to cherish and rescue the contribution of important women in STEM (science, technology, engineering and mathematics).


Congratulations in completing the final challenge of this programming quest! You can access the final page of this quest by link adding the following words to the link:
aacoltfp


"""

cipher_text = vigenere_cipher(section, key= "PURPLE".lower())
print(cipher_text.upper())


PXR LYVOJIL, WFJYXTMJ ZJ FFKPPPWV, APM ISI ZZGDX JVGDSC KD IKYI HVXNV L ELFVCEB. SLYVBKTC DZ ISI MBXWPTX BLXWYDPEMRCRC ECHRQPPAU BTPQUEZP LCKW XWY EZII CDCH VPGZR, RSL HBFLPH YEISYHCRHX UII XEIBVBLXXWJ LRS JRTICWV, FJN PWWD RGEW UES QJMZR. HNZBFPPNVS FN YTC BIKWPV, NWZ HBVAEIGYU SIG WGZQ NYT IMWVHDIH RCO XGDDCEA STSEKCFG SU YTC UUKWPV, RSL VLVL YE NXEL U HEVDHX LRPFPITGPF BTRS. LSIC JWP LUJ 12 SVPCW ICS, HBV HEH ZCDTXLVS FN KWP XXVP SU WAJMCA, LRS VMEICMZKPPN JIFHXYU MMGXJ LRS RTCSSSEPXMRM. LX 17, LTL EZXTHKXLP UJ L BUKWPQPNZRTEC JILVIYU ES VV CIRIXCTDTX.

MC YTC ACWT, HBV SES TDYXPWK HMIB PYH GVI QPHP TQEIIILRI JRTICNZUTG ZZVFVTM. LQDHX ELTG, HEH TWLVAYJ MEQVRVP, QYDDI CEKPRICFCD LYIT SU XGPEI ZCEIGYJI XD RSL. UUP YCXVGDXDIU ELT DTNLPHZHXW VVWTRS SPMFPAV’D RIDEFXXHX PRVCETD PHU HEH WPDGXHRIPH VP ELTG. PZTHKJLPAS, LHP TDYXGCSJEIS KD XWY ICECMCPEMDH DQ P KTIX USDFX VRQLKT’J LRPFPITGPF TYKXHV LRS TDFPS NGTXT YTC DQE AIGMFCLP HFIPW USDFX NYT QPNKTC. BVG RDNVH ICXVS YE STTRV DDCI YOIPRHCMT